# Instructor guide: contribution workflow

Use after the learner has made predictions. [Learner notebook](../00-contribution-workflow.ipynb) · [Case guide](../../docs/contribution-case.md) · [Observation protocol](../../docs/contribution-walkthrough-pilot.md)

This is an AI-assisted local case based on issue #17113 and PR #17117, not an upstream patch or evidence of human outcomes. Do not turn these answers into a learner-authored PR. The current project contribution policy remains authoritative.

## Before the session

Verify the two release commands in the case guide yourself; don't install the old release into QCLAB's environment. Allow a separate setup check and a provisional 60–90-minute session. Obtain note-taking consent, record previous source/fix exposure and all help, and retain incomplete attempts. No participant records go into this repository.

A missing download, unsupported wheel, or source-build failure is an environment blocker, not the historical numerical bug. The notebook shows the fixed wheel. Neither historical checkout has been built as part of these materials.

In [ ]:
import sys

import qiskit

print("Python:", sys.version.split()[0])
print("Qiskit:", qiskit.__version__)
print("Expected teaching release: 2.6.2 (fixed). Historical comparison: 2.5.2 and 2.6.0.")

## Expected reasoning, not a script for the learner

For H = 0 or t = 0, exp(-i t H) is identity, so the rotation expansion may be empty without numerical warnings. In 2.5.2, zero operator weight causes invalid normalization; zero time can return an empty list while still performing invalid arithmetic. A result-only zero-time test misses this problem.

This supports a guard before the divisions in Python `QDrift.expand`. It does not justify editing Rust or returning an empty list for every input. Ask the learner to locate the implementation and tests themselves before showing the fixed diff.

In [ ]:
import warnings

from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp
from qiskit.synthesis import QDrift

# Observe the installed release; no monkeypatch or simulated old failure.
zero_evolution = PauliEvolutionGate(SparsePauliOp("Z", coeffs=[0.0]), time=2.0)
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter("always")
    try:
        expansion = QDrift(seed=7).expand(zero_evolution)
        print("Zero-operator expansion:", expansion)
    except (ValueError, FloatingPointError, ZeroDivisionError) as exc:
        print("Observed exception:", type(exc).__name__, str(exc))
print("Warnings:", len(observed_warnings))
for warning in observed_warnings:
    print(warning.category.__name__, str(warning.message))

## One local reference test design

The following independently written teaching tests use a different Pauli term, coefficient/time choices, and seed from the upstream examples. They exercise the public API and make NumPy's invalid arithmetic fail explicitly. Expected behavior comes from identity evolution, not recomputing the algorithm. On 2.5.2 the first two cases fail; on 2.6.0 they pass. The nonzero case guards against a trivial always-empty implementation, but does not establish randomized multi-term accuracy.

In [ ]:
import numpy as np

for coefficient, time in [(0.0, 2.0), (1.0, 0.0)]:
    gate = PauliEvolutionGate(SparsePauliOp("Z", coeffs=[coefficient]), time=time)
    with np.errstate(divide="raise", invalid="raise"):
        assert QDrift(seed=7).expand(gate) == []

control = PauliEvolutionGate(SparsePauliOp("Z", coeffs=[1.0]), time=1.0)
with np.errstate(divide="raise", invalid="raise"):
    assert QDrift(seed=7).expand(control) == [("Z", [0], 1.0), ("Z", [0], 1.0)]
print("Reference API regressions passed on the installed release.")

## Source map and limits

At fixed commit `65094a013073e4bda53c2b341056ab9ade0eb1bd`:

- `qiskit/synthesis/evolution/qdrift.py`: Python `QDrift.expand`, including the early zero-case return.
- `test/python/circuit/library/test_evolution_gate.py`: upstream Python tests for zero time and zero operator.
- `qiskit/synthesis/evolution/product_formula.py`: Python `synthesize` calls `self.expand`, then the default path calls `pauli_evolution` imported from `_accelerate.circuit_library`.
- `crates/circuit_library/src/pauli_evolution.rs`: Rust `py_pauli_evolution` binding; registration is in the adjacent `lib.rs`.

[Exact linked evidence](../../docs/walkthrough-case-research.md). `expand` itself does not exercise the later Rust call. Wheel comparisons are not isolated patch tests; the exact historical diff establishes what changed. Understanding these paths does not establish Rust proficiency.

## Human assessment — revision 1

Material version: contribution-workflow-r1-2026-11-01. Keep this guide aside during the initial attempt. It is publicly accessible, not a secret assessment. Ask about prior exposure before selecting a scenario. If it was seen or discussed already, select a different scenario or record this as a rehearsal; do not call it fresh.

Suggested follow-up: A proposed patch returns an empty expansion whenever time is less than 0.01. It passes both zero-case tests and the existing nonzero control at time 1.0. Is that sufficient? Describe a counterexample test without writing the implementation.

Facilitator reasoning: the threshold also suppresses valid nonzero evolutions. A single nonzero Pauli with positive time below 0.01, such as coefficient 1 and time 0.005, should still have a nonempty expansion. A targeted test rejects that patch. This counterexample does not prove general randomized accuracy. Do not supply the counterexample before recording the first answer.

If that scenario is already familiar, devise a different bounded change and record its wording and expected rationale before asking it. Do not turn questions already answered in the notebook (warnings, wheel versus source, or source navigation) into claims of fresh transfer. The signed-coefficient question printed in the original learner notebook is a discussion prompt, not a fresh assessment for anyone exposed to that version.

Record the initial answer, each subsequent prompt, and the final answer separately. Count assistance on navigation and assistance on the follow-up as separate episodes. Label unaided / documentation-only / assisted / not demonstrated separately for workflow, navigation, reproduction, regression rationale, and explanation. Preserve the learner's initial test sketch and references used during refinement; different Pauli terms or seeds alone do not establish independent design. No single answer establishes contribution readiness or Rust proficiency.

In [ ]:
print("Walkthrough execution complete.")
print("Human review: pending")
print("No automatic contribution-readiness grade is assigned.")